In [3]:
# %% [markdown]
# RQ2 — Step 0 (No cutoff at all): Full clone + fetch only
# Reads a CSV with column `repo_url`, clones/fetches into CLONE_ROOT,
# promotes any shallow/partial repo to full, fetches tags & all branches,
# (optionally PR heads, submodules, LFS), and writes a manifest.
# NOTE: No cutoff branches are created and no cutoff checks are performed.

# %%
from __future__ import annotations
import csv, subprocess, time
from pathlib import Path
from typing import Optional, List

# -----------------------------
# Config (edit as needed)
# -----------------------------

WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"          # input CSV with column 'repo_url'
CLONE_ROOT   = WORK_ROOT / "clonesV7.0"            # repos will clone here
MANIFEST_CSV = WORK_ROOT / "clones_manifestV7.0.csv"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# Toggle if you need submodules or LFS contents for analysis
WITH_SUBMODULES = False
WITH_LFS        = False

# %%
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)

def sh_ok(cmd: List[str], cwd: Optional[Path] = None) -> str:
    cp = sh(cmd, cwd=cwd, check=True)
    return cp.stdout

def repo_dir_name_from_url(url: str) -> str:
    # e.g. https://github.com/owner/name(.git) -> owner__name
    base = url.split("//")[-1]
    parts = base.split("/")
    if len(parts) >= 3:
        owner = parts[-2]
        name  = parts[-1].replace(".git", "")
        return f"{owner}__{name}"
    return base.replace("/", "__").replace(".git", "")

def ensure_full_clone(url: str, dest_root: Path,
                      with_submodules: bool = False,
                      with_lfs: bool = False) -> Path:
    """
    Ensures a FULL, analysis-ready clone:
      - full history (no filters)
      - tags fetched
      - all remote branches present (origin/*)
      - best-effort PR heads (GitHub)
      - optional submodules + LFS
    Promotes existing shallow/partial clones in-place.
    """
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)

    if not (d.exists() and (d / ".git").exists()):
        # Fresh FULL clone (no partial/shallow flags)
        sh_ok(["git", "clone", "--no-recurse-submodules", "--tags", url, str(d)])
    else:
        # Make sure origin URL is correct
        try:
            sh_ok(["git", "remote", "set-url", "origin", url], cwd=d)
        except Exception:
            pass

    # If shallow, unshallow to get complete history
    try:
        cp = sh(["git", "rev-parse", "--is-shallow-repository"], cwd=d, check=False)
        if cp.returncode == 0 and cp.stdout.strip() == "true":
            sh_ok(["git", "fetch", "--unshallow", "--tags"], cwd=d)
    except Exception:
        # If the check failed, just ensure we fetched tags anyway
        sh_ok(["git", "fetch", "--tags"], cwd=d)

    # Fetch all remote branches (promotes partial clones, fills objects)
    sh_ok([
        "git", "fetch", "origin", "--prune", "--tags",
        "+refs/heads/*:refs/remotes/origin/*"
    ], cwd=d)

    # (Optional) Include PR heads if you study pre-merge history on GitHub
    sh(["git", "fetch", "origin",
        "+refs/pull/*/head:refs/remotes/origin/pr/*"], cwd=d, check=False)

    if with_submodules:
        sh_ok(["git", "submodule", "update", "--init", "--recursive"], cwd=d)

    if with_lfs:
        try:
            sh_ok(["git", "lfs", "install"], cwd=d)
            sh_ok(["git", "lfs", "fetch", "--all"], cwd=d)
            sh_ok(["git", "lfs", "checkout"], cwd=d)
        except Exception:
            pass

    return d

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir, check=False)
    return int((cp.stdout or "0").strip() or "0")

# %%
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {
            "repo_url": url,
            "dir": None,
            "status": "unknown",
            "seconds": None,
            "total_commits": None,
            "cutoff_branches": "",  # kept for schema compatibility; always blank
            "error": ""
        }
        try:
            d = ensure_full_clone(url, CLONE_ROOT,
                                  with_submodules=WITH_SUBMODULES,
                                  with_lfs=WITH_LFS)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)

            # No cutoff branch creation or checks here.

            rec["status"] = "ok"
            ok += 1

        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        except Exception as e:
            rec["status"] = "error"
            rec["error"]  = str(e)[:2000]
            fail += 1

        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        # keep the same print shape; cutoff_branches will be '-' in the log, but status will be ok
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)  "
              f"cutoff_branches={rec['cutoff_branches'] or '-'}")

# %%
# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ["repo_url","dir","status","seconds","total_commits","cutoff_branches","error"]
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")
print("No cutoff logic: clone + fetch only (no cutoff/* branches created).")


[ok] https://github.com/inaturalist/iNaturalistAndroid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV7.0\inaturalist__iNaturalistAndroid (0.88s)  cutoff_branches=-
[ok] https://github.com/BlueWallet/BlueWallet -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV7.0\BlueWallet__BlueWallet (8.28s)  cutoff_branches=-
[ok] https://github.com/haiyangwu/mediasoup-client-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV7.0\haiyangwu__mediasoup-client-android (0.52s)  cutoff_branches=-

Done. OK=3, FAIL=0. Manifest: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones_manifestV7.0.csv
No cutoff logic: clone + fetch only (no cutoff/* branches created).
